# Convolutional Neural Networks

# Importar Librerías

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
import keras
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential,Model
from tensorflow.keras.layers import Input
from keras.layers import Dense, Dropout, Flatten
from keras.layers import LeakyReLU

# Cargar set de Imágenes

In [ ]:
from pathlib import Path

root_path = Path.cwd() / 'animales'
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

images = []
directories = []
dircount = []

print(f"Leyendo imágenes de {root_path}")

for subdir in sorted(root_path.rglob('*')):
    if subdir.is_dir():
        valid_files = [f for f in subdir.iterdir() if f.suffix.lower() in extensions]
        
        if valid_files:
            current_dir_count = 0
            for file_path in valid_files:
                image = plt.imread(str(file_path))
                
                if image.ndim == 3:
                    images.append(image)
                    current_dir_count += 1
            
            if current_dir_count > 0:
                directories.append(str(subdir))
                dircount.append(current_dir_count)

print('Directorios leídos:', len(directories))
print("Imágenes en cada directorio:", dircount)
print('Suma total de imágenes:', sum(dircount))

# Creamos las etiquetas

In [ ]:
labels=[]
indice=0
for cantidad in dircount:
    for i in range(cantidad):
        labels.append(indice)
    indice=indice+1
print("Cantidad etiquetas creadas: ",len(labels))


In [ ]:
animales=[]
indice=0
for directorio in directories:
    name = directorio.split(os.sep)
    print(indice , name[-1])
    animales.append(name[-1])
    indice=indice+1

In [ ]:
y = np.array(labels)
X = np.array(images, dtype=np.uint8) #convierto de lista a numpy



# Find the unique numbers from the train labels
classes = np.unique(y)
nClasses = len(classes)
print('Total number of outputs : ', nClasses)
print('Output classes : ', classes)

# Creamos Sets de Entrenamiento y Test

In [ ]:
train_X,test_X,train_Y,test_Y = train_test_split(X,y,test_size=0.2)
print('Training data shape : ', train_X.shape, train_Y.shape)
print('Testing data shape : ', test_X.shape, test_Y.shape)

In [ ]:
plt.figure(figsize=[5,5])

# Display the first image in training data
plt.subplot(121)
plt.imshow(train_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(train_Y[0]))

# Display the first image in testing data
plt.subplot(122)
plt.imshow(test_X[0,:,:], cmap='gray')
plt.title("Ground Truth : {}".format(test_Y[0]))

# Preprocesamos las imagenes

In [ ]:
train_X = train_X.astype('float32')
test_X = test_X.astype('float32')
train_X = train_X/255.
test_X = test_X/255.
plt.imshow(test_X[0,:,:])

## Hacemos el One-hot Encoding para la red

In [ ]:
# Change the labels from categorical to one-hot encoding
train_Y_one_hot = to_categorical(train_Y)
test_Y_one_hot = to_categorical(test_Y)

# Display the change for category label using one-hot encoding
print('Original label:', train_Y[0])
print('After conversion to one-hot:', train_Y_one_hot[0])

# Creamos el Set de Entrenamiento y Validación

In [ ]:
#Mezclar todo y crear los grupos de entrenamiento y testing
train_X,valid_X,train_label,valid_label = train_test_split(train_X, train_Y_one_hot, test_size=0.2, random_state=48)

In [ ]:
print(train_X.shape,valid_X.shape,train_label.shape,valid_label.shape)

# Creamos el modelo de CNN

In [ ]:
#declaramos variables con los parámetros de configuración de la red
INIT_LR = 1e-3 # Valor inicial de learning rate. El valor 1e-3 corresponde con 10
epochs = 20 # Cantidad de iteraciones completas al conjunto de imagenes de entrenamiento
batch_size = 16 # cantidad de imágenes que se toman a la vez en memoria

In [ ]:
animales_model = Sequential()
animales_model.add(Conv2D(32, kernel_size=(3, 3),activation='linear', padding='same', input_shape=(28,21,3)))
animales_model.add(LeakyReLU(alpha=0.1))
animales_model.add(MaxPooling2D((2, 2),padding='same'))
animales_model.add(Dropout(0.5))
animales_model.add(Flatten())
animales_model.add(Dense(32, activation='linear'))
animales_model.add(LeakyReLU(alpha=0.1))
animales_model.add(Dropout(0.5))
animales_model.add(Dense(nClasses, activation='softmax'))

In [ ]:
animales_model.summary()

In [ ]:
animales_model.compile(loss=keras.losses.categorical_crossentropy, optimizer=tf.keras.optimizers.SGD(learning_rate=INIT_LR),metrics=['accuracy'])

# Entrenamos el modelo: Aprende a clasificar imágenes

In [ ]:
# este paso puede tomar varios minutos, dependiendo de tu ordenador, cpu y memoria ram libre
animales_train = animales_model.fit(train_X, train_label, batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(valid_X, valid_label))

In [ ]:
# guardamos la red, para reutilizarla en el futuro, sin tener que volver a entrenar
import json

with open("state.json", 'r') as file:
    data = json.load(file)
animales_model.save(f"modelos/animales_{data['model_index']}.keras")

data['model_index']+=1

with open("state.json", 'w') as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

# Evaluamos la red

In [ ]:
test_eval = animales_model.evaluate(test_X, test_Y_one_hot, verbose=1)

In [ ]:
print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

In [ ]:
animales_train.history

In [ ]:
accuracy = animales_train.history['accuracy']
val_accuracy = animales_train.history['val_accuracy']
loss = animales_train.history['loss']
val_loss = animales_train.history['val_loss']
epochs = range(len(accuracy))
plt.plot(epochs, accuracy, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
predicted_classes2 = animales_model.predict(test_X)

In [ ]:
predicted_classes=[]
for predicted_animales in predicted_classes2:
    predicted_classes.append(predicted_animales.tolist().index(max(predicted_animales)))
predicted_classes=np.array(predicted_classes)

In [ ]:
predicted_classes.shape, test_Y.shape

# Aprendamos de los errores: Qué mejorar

In [ ]:
correct = np.where(predicted_classes==test_Y)[0]
print("Found %d correct labels" % len(correct))
for i, correct in enumerate(correct[0:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(test_X[correct].reshape(21,28,3), cmap='gray', interpolation='none')
    plt.title("{}, {}".format(animales[predicted_classes[correct]],
                                                    animales[test_Y[correct]]))

    plt.tight_layout()

In [ ]:
incorrect = np.where(predicted_classes!=test_Y)[0]
print("Found %d incorrect labels" % len(incorrect))
for i, incorrect in enumerate(incorrect[0:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(test_X[incorrect].reshape(21,28,3), cmap='gray', interpolation='none')
    plt.title("{}, {}".format(animales[predicted_classes[incorrect]],
                                                    animales[test_Y[incorrect]]))
    plt.tight_layout()

In [ ]:
target_names = ["Class {}".format(i) for i in range(nClasses)]
print(classification_report(test_Y, predicted_classes, target_names=target_names))

In [ ]:
from skimage.transform import resize
animales_modelos = keras.models.load_model("modelos/animales_1.keras")

images=[]
# AQUI ESPECIFICAMOS UNAS IMAGENES
filenames = list(Path("test_images").glob("*.png")) + list(Path("test_images").glob("*.jpg")) + list(Path("test_images").glob("*.jpeg"))

for filepath in filenames:
    try:
        image = plt.imread(filepath)
        image_resized = resize(image, (28, 21),anti_aliasing=True,clip=False,preserve_range=True)

        if image_resized.shape[-1] == 4:
            image_resized = image_resized[:, :, :3]

        images.append(image_resized)
    except Exception as e:
        print(f"{filepath}:{e}")


X = np.array(images, dtype=np.uint8) #convierto de lista a numpy
test_X = X.astype('float32')
test_X = test_X / 255.

predicted_classes = animales_model.predict(test_X)

for i, pred in enumerate(predicted_classes):
    print(f"\nImagen: {filenames[i]}")
    
    for j, prob in enumerate(pred):
        print(f"{animales[j]}: {prob:.4f}")
    
    clase_pred = np.argmax(pred)
    print(f"Final: {animales[clase_pred]}")
